# Somina WooCommerce -- Ventas Netas Check (Resumen vs Productos)

This notebook connects directly to the WooCommerce MySQL database, pulls the same
data behind the *Resumen* and *Productos* reports, and reconciles them in pandas --
independent of whatever the Analytics cache currently shows in the dashboard.

Set the date range in section 2 and run the cells top to bottom.

In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine
from dotenv import load_dotenv

pd.set_option("display.float_format", lambda x: f"{x:,.0f}")

## 1. Connect to the database

Credentials come from a local `.env` file, never hardcoded here -- see
`.env.example` next to this notebook. Copy it to `.env`, fill in your real
values, and keep `.env` out of version control (add it to `.gitignore`).

If you're running this notebook from your own machine rather than on the
server itself, you'll likely need to enable **Remote MySQL** access for this
database in Hostinger's hPanel and allow your current IP -- shared hosting
only accepts local connections by default.

In [2]:
load_dotenv()

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "3306")
DB_NAME = os.getenv("DB_NAME")

engine = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

with engine.connect() as conn:
    print("Connected OK")

Connected OK


## 2. Set the period to check

Change these two dates and every query below updates automatically.

In [3]:
START_DATE = "2026-08-01 00:00:00"
END_DATE   = "2026-08-08 23:59:59"

## 3. Orders recalculated directly from `wp_posts` / `wp_postmeta`

Bypasses the Analytics cache entirely and rebuilds order totals from the
actual order records (this store uses classic/CPT storage, confirmed earlier).

In [4]:
query_orders = f"""
SELECT
    p.ID AS order_id,
    p.post_status AS status,
    p.post_date_gmt AS date_created_gmt,
    MAX(CASE WHEN pm.meta_key = '_order_total'    THEN pm.meta_value END) + 0 AS order_total,
    MAX(CASE WHEN pm.meta_key = '_order_tax'      THEN pm.meta_value END) + 0 AS order_tax,
    MAX(CASE WHEN pm.meta_key = '_order_shipping' THEN pm.meta_value END) + 0 AS order_shipping,
    (
        MAX(CASE WHEN pm.meta_key = '_order_total'    THEN pm.meta_value END) -
        MAX(CASE WHEN pm.meta_key = '_order_tax'      THEN pm.meta_value END) -
        MAX(CASE WHEN pm.meta_key = '_order_shipping' THEN pm.meta_value END)
    ) + 0 AS net_sales_recalculated
FROM wp_posts p
JOIN wp_postmeta pm ON pm.post_id = p.ID
WHERE p.post_type = 'shop_order'
  AND p.post_status NOT IN ('wc-pending', 'wc-cancelled', 'wc-failed', 'trash')
  AND p.post_date_gmt BETWEEN '{START_DATE}' AND '{END_DATE}'
GROUP BY p.ID
"""

df_orders = pd.read_sql(query_orders, engine)
df_orders

,order_id,status,date_created_gmt,order_total,order_tax,order_shipping,net_sales_recalculated
0,4913,wc-completed,2026-08-08 14:55:31,"240,000",0,0,"240,000"
1,4923,wc-completed,2026-08-03 15:53:49,"43,000",0,0,"43,000"
2,4924,wc-completed,2026-08-03 17:09:25,"329,300",0,0,"329,300"
3,4925,wc-completed,2026-08-03 19:01:37,"96,000",0,0,"96,000"
4,4927,wc-completed,2026-08-04 23:50:11,"170,600",0,0,"170,600"
5,4929,wc-completed,2026-08-05 19:29:36,"15,504",0,0,"15,504"
6,4931,wc-completed,2026-08-05 19:33:10,"136,500",0,0,"136,500"
7,4932,wc-completed,2026-08-05 19:42:25,"72,000",0,0,"72,000"
8,4936,wc-completed,2026-08-07 01:01:23,"240,400",0,0,"240,400"
9,4937,wc-completed,2026-08-08 21:58:01,"77,297",0,0,"77,297"


## 4. Pull the Analytics cache (`wc_order_stats`) for the same period

This is what actually feeds the *Resumen* report in the dashboard.

In [ ]:
query_stats = f"""
SELECT order_id, status, date_created_gmt, total_sales, net_total
FROM wp_wc_order_stats
WHERE date_created_gmt BETWEEN '{START_DATE}' AND '{END_DATE}'
"""

df_stats = pd.read_sql(query_stats, engine)
df_stats

## 5. Product-level report with SKU (`wc-completed` orders)

In [ ]:
query_products = f"""
SELECT
    pl.product_id,
    prod.post_title AS producto,
    sku.meta_value AS sku,
    SUM(pl.product_qty) AS unidades_vendidas,
    SUM(pl.product_net_revenue) AS ventas_netas,
    SUM(pl.product_gross_revenue) AS ventas_brutas
FROM wp_wc_order_product_lookup pl
JOIN wp_posts ord ON ord.ID = pl.order_id
LEFT JOIN wp_posts prod ON prod.ID = pl.product_id
LEFT JOIN wp_postmeta sku ON sku.post_id = pl.product_id AND sku.meta_key = '_sku'
WHERE ord.post_status = 'wc-completed'
  AND pl.date_created BETWEEN '{START_DATE}' AND '{END_DATE}'
GROUP BY pl.product_id, prod.post_title, sku.meta_value
ORDER BY ventas_netas DESC
"""

df_products = pd.read_sql(query_products, engine)
df_products

## 6. Reconciliation: which order(s) explain the gap?

Sum each order's own line items from `wc_order_product_lookup`, then compare
against what `wc_order_stats` reports for that same order.

In [ ]:
query_product_sums = f"""
SELECT order_id, SUM(product_net_revenue) AS ventas_netas_productos
FROM wp_wc_order_product_lookup
WHERE date_created BETWEEN '{START_DATE}' AND '{END_DATE}'
GROUP BY order_id
"""

df_product_sums = pd.read_sql(query_product_sums, engine)

df_reconciliation = df_stats.merge(df_product_sums, on="order_id", how="left")
df_reconciliation["ventas_netas_productos"] = df_reconciliation["ventas_netas_productos"].fillna(0)
df_reconciliation["diferencia"] = df_reconciliation["net_total"] - df_reconciliation["ventas_netas_productos"]

df_reconciliation.sort_values("diferencia", key=abs, ascending=False)

## 7. Quick visual: Resumen vs Productos, per order

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
x = df_reconciliation["order_id"].astype(str)
width = 0.35
positions = range(len(x))
ax.bar([i - width/2 for i in positions], df_reconciliation["net_total"], width, label="Resumen (cache)")
ax.bar([i + width/2 for i in positions], df_reconciliation["ventas_netas_productos"], width, label="Productos (cache)")
ax.set_xticks(list(positions))
ax.set_xticklabels(x, rotation=45)
ax.set_ylabel("Ventas netas")
ax.set_title(f"Resumen vs Productos por pedido ({START_DATE[:10]} a {END_DATE[:10]})")
ax.legend()
plt.tight_layout()
plt.show()

## 8. Export the final report

In [ ]:
from datetime import date

output_path = f"resumen_woocommerce_{date.today().isoformat()}.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    df_orders.to_excel(writer, sheet_name="Pedidos recalculados", index=False)
    df_stats.to_excel(writer, sheet_name="Resumen (cache)", index=False)
    df_products.to_excel(writer, sheet_name="Productos", index=False)
    df_reconciliation.to_excel(writer, sheet_name="Reconciliacion", index=False)

print(f"Reporte guardado en: {output_path}")